# Adapter trim — cutadapt + fastp

Вырезает Illumina TruSeq/Nextera адаптер, фильтрует Q30 + minlen 250.
Вход: сырые FASTQ из `raw/<DS>/`.
Выход: `results/<DS>/trimmed/{fastq, fastp_reports}/`.

In [11]:
import os, sys
_CONDA_ENV = "/opt/conda/envs/bcr_env"
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
if os.path.isdir(_CONDA_ENV + "/lib/python3.11/site-packages"):
    sys.path.insert(0, _CONDA_ENV + "/lib/python3.11/site-packages")

In [12]:
import shutil, subprocess
from pathlib import Path

# Адаптеры по датасетам. Для "auto" cutadapt запустится с --detect-adapter,
# а fastp всё равно дотримливает через --detect_adapter_for_pe.
ADAPTERS = {
    "PRJEB40348": "CTGTCTCTTATACACATCT",  # Nextera (human) — подтверждено fastp detect
    "PRJNA848968": "auto",
    "PRJNA1247978": "auto",
    "PRJNA900592": "auto",
}

In [13]:
def run_adapter_trim(volume, dataset):
    vol = Path(volume)
    src = vol / "raw" / dataset
    base = vol / "results" / dataset / "trimmed"

    cut = base / "cutadapt"
    out = base / "fastq"
    rpt = base / "fastp_reports"

    cut.mkdir(parents=True, exist_ok=True)
    out.mkdir(parents=True, exist_ok=True)
    rpt.mkdir(parents=True, exist_ok=True)

    adapter = ADAPTERS.get(dataset, "auto")

    pairs = sorted(set(
        f.name.rsplit("_", 1)[0] for f in src.glob("*.fastq.gz")
    ))
    print(f"[adapter_trim] {dataset}: {len(pairs)} pairs (adapter={adapter})")

    for bn in pairs:
        r1 = src / f"{bn}_1.fastq.gz"
        r2 = src / f"{bn}_2.fastq.gz"
        if not r1.exists():
            continue
        if (out / f"{bn}_1.trim.fastq.gz").exists():
            print(f"  {bn} already done, skip")
            continue

        c1 = cut / f"{bn}_1.cut.fastq.gz"
        c2 = cut / f"{bn}_2.cut.fastq.gz"

        # cutadapt: адаптер + качественная обрезка 3'-конца (Q<30)
        ca_cmd = ["cutadapt", "--quality-cutoff", "30", "--compression-level", "1"]
        if adapter == "auto":
            ca_cmd += ["--detect-adapter", "p=2"]  # авто-поиск адаптера (PE)
        else:
            ca_cmd += ["-a", adapter, "-A", adapter]
        ca_cmd += ["-o", str(c1), "-p", str(c2), str(r1), str(r2)]

        print(f"  [{bn}] cutadapt ...")
        subprocess.run(ca_cmd, capture_output=True)

        # fastp: дотримливает адаптер (auto), обрезка Q<30, фильтр minlen 250
        print(f"  [{bn}] fastp ...")
        subprocess.run(["fastp",
                        "-i", str(c1), "-I", str(c2),
                        "-o", str(out / f"{bn}_1.trim.fastq.gz"),
                        "-O", str(out / f"{bn}_2.trim.fastq.gz"),
                        "--detect_adapter_for_pe",
                        "-q", "30", "-l", "250", "-w", "4",
                        "-h", str(rpt / f"{bn}.html"),
                        "-j", str(rpt / f"{bn}.json")], capture_output=True)

        c1.unlink(missing_ok=True)
        c2.unlink(missing_ok=True)

    if cut.exists():
        shutil.rmtree(cut)
    print(f"[adapter_trim] DONE: {len(list(out.glob('*.trim.fastq.gz')))} files")


### Запуск

PRJEB40348, PRJNA848968, PRJNA1247978, PRJNA900592.

In [14]:
run_adapter_trim("/data/user/epishkin", "PRJEB40348")

[adapter_trim] PRJEB40348: 35 pairs (adapter=CTGTCTCTTATACACATCT)
  [ERR4588253] cutadapt ...
  [ERR4588253] fastp ...
  [ERR4588254] cutadapt ...
  [ERR4588254] fastp ...
  [ERR4588255] cutadapt ...
  [ERR4588255] fastp ...
  [ERR4588256] cutadapt ...
  [ERR4588256] fastp ...
  [ERR4588257] cutadapt ...
  [ERR4588257] fastp ...
  [ERR4588258] cutadapt ...
  [ERR4588258] fastp ...
  [ERR4588259] cutadapt ...
  [ERR4588259] fastp ...
  [ERR4588260] cutadapt ...
  [ERR4588260] fastp ...
  [ERR4588261] cutadapt ...
  [ERR4588261] fastp ...
  [ERR4588262] cutadapt ...
  [ERR4588262] fastp ...
  [ERR4588263] cutadapt ...
  [ERR4588263] fastp ...
  [ERR4588264] cutadapt ...
  [ERR4588264] fastp ...
  [ERR4588265] cutadapt ...
  [ERR4588265] fastp ...
  [ERR4588266] cutadapt ...
  [ERR4588266] fastp ...
  [ERR4588267] cutadapt ...
  [ERR4588267] fastp ...
  [ERR4588268] cutadapt ...
  [ERR4588268] fastp ...
  [ERR4588269] cutadapt ...
  [ERR4588269] fastp ...
  [ERR4588270] cutadapt ...
  [ER

In [16]:
run_adapter_trim("/data/user/epishkin", "PRJNA848968")

[adapter_trim] PRJNA848968: 4 pairs (adapter=auto)
  [SRR19646177] cutadapt ...
  [SRR19646177] fastp ...
  [SRR19646178] cutadapt ...
  [SRR19646178] fastp ...
  [SRR19646179] cutadapt ...
  [SRR19646179] fastp ...
  [SRR19646180] cutadapt ...
  [SRR19646180] fastp ...
[adapter_trim] DONE: 0 files


In [17]:
run_adapter_trim("/data/user/epishkin", "PRJNA1247978")

[adapter_trim] PRJNA1247978: 60 pairs (adapter=auto)
  [SRR33022240] cutadapt ...
  [SRR33022240] fastp ...
  [SRR33022241] cutadapt ...
  [SRR33022241] fastp ...
  [SRR33022242] cutadapt ...
  [SRR33022242] fastp ...
  [SRR33022243] cutadapt ...
  [SRR33022243] fastp ...
  [SRR33022244] cutadapt ...
  [SRR33022244] fastp ...
  [SRR33022245] cutadapt ...
  [SRR33022245] fastp ...
  [SRR33022246] cutadapt ...
  [SRR33022246] fastp ...
  [SRR33022247] cutadapt ...
  [SRR33022247] fastp ...
  [SRR33022248] cutadapt ...
  [SRR33022248] fastp ...
  [SRR33022249] cutadapt ...
  [SRR33022249] fastp ...
  [SRR33022250] cutadapt ...
  [SRR33022250] fastp ...
  [SRR33022251] cutadapt ...
  [SRR33022251] fastp ...
  [SRR33022252] cutadapt ...
  [SRR33022252] fastp ...
  [SRR33022253] cutadapt ...
  [SRR33022253] fastp ...
  [SRR33022254] cutadapt ...
  [SRR33022254] fastp ...
  [SRR33022255] cutadapt ...
  [SRR33022255] fastp ...
  [SRR33022256] cutadapt ...
  [SRR33022256] fastp ...
  [SRR330222

In [18]:
run_adapter_trim("/data/user/epishkin", "PRJNA900592")

[adapter_trim] PRJNA900592: 12 pairs (adapter=auto)
  [SRR22279249] cutadapt ...
  [SRR22279249] fastp ...
  [SRR22279250] cutadapt ...
  [SRR22279250] fastp ...
  [SRR22279251] cutadapt ...
  [SRR22279251] fastp ...
  [SRR22279252] cutadapt ...
  [SRR22279252] fastp ...
  [SRR22279253] cutadapt ...
  [SRR22279253] fastp ...
  [SRR22279254] cutadapt ...
  [SRR22279254] fastp ...
  [SRR22279255] cutadapt ...
  [SRR22279255] fastp ...
  [SRR22279256] cutadapt ...
  [SRR22279256] fastp ...
  [SRR22279257] cutadapt ...
  [SRR22279257] fastp ...
  [SRR22279258] cutadapt ...
  [SRR22279258] fastp ...
  [SRR22279259] cutadapt ...
  [SRR22279259] fastp ...
  [SRR22279260] cutadapt ...
  [SRR22279260] fastp ...
[adapter_trim] DONE: 0 files
